In [18]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import shape
from pathlib import Path
import os
from bs4 import BeautifulSoup
import re

In [19]:
CURRENT_DIRECTORY =  os.getcwd()
SRC_DIRECTORY = Path(CURRENT_DIRECTORY).parents[1]
print(SRC_DIRECTORY)

BASE_DATASET_PATH = Path(SRC_DIRECTORY).parents[0]
BASE_DATASET_PATH = BASE_DATASET_PATH / "datasets"
print(BASE_DATASET_PATH)

MISSING_VALUES = ["nan", "-", "NaN", 'N/A', 'NA', "na"]

/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/src
/Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets


In [20]:
def convert_cols_to_int(df):
    """
    convert the values of columns other than planning_area and subzone into int. 
    """

    exclude_cols = ['planning_area', 'subzone']

    columns_to_convert = [col for col in df.columns if col not in exclude_cols]

    # Clean and Convert the columns
    for col in columns_to_convert:
        # A. Fill any remaining true NaN values with 0. 
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
        # B. Cast the cleaned (now float) column to the integer type
        df[col] = df[col].astype('int64')

In [21]:
ethnicity_2010_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2010_ethnicity_subzone_edited.xlsx")
ethnicity_2015_filepath = Path(BASE_DATASET_PATH / "singapore_data/data_gov/2015_ethnicity_subzone_edited.xlsx")
ethnicity_2020_filepath = Path(BASE_DATASET_PATH / "singapore_data/singstat/2020_ethnicity_subzone_edited.xlsx")

ethnicity_2010 = pd.read_excel(ethnicity_2010_filepath)
ethnicity_2015 = pd.read_excel(ethnicity_2015_filepath)
ethnicity_2020 = pd.read_excel(ethnicity_2020_filepath)


In [22]:
columns_to_lower = ["planning_area", "subzone"]
ethnicity_2010.columns

Index(['planning_area', 'subzone', 'total', 'male_total', 'female_total',
       'total_chinese', 'male_chinese', 'female_chinese', 'total_malays',
       'male_malays', 'female_malays', 'total_indians', 'male_indians',
       'female_indians', 'total_others', 'male_others', 'female_others'],
      dtype='object')

In [23]:
# make the planning_area and subzone names lowercaps
ethnicity_2010[columns_to_lower] = ethnicity_2010[columns_to_lower].apply(lambda x: x.str.lower().str.strip(), 
                                                                          axis = 0)
# remove the - total trailing string from planning_area column
ethnicity_2010['planning_area'] = ethnicity_2010['planning_area'].str.replace('- total', '').str.strip()
ethnicity_2010.head()

,planning_area,subzone,total,male_total,female_total,total_chinese,male_chinese,female_chinese,total_malays,male_malays,female_malays,total_indians,male_indians,female_indians,total_others,male_others,female_others
0,total,total,3771721,1861133,1910588,2793980,1370083,1423897,503868,250885,252983,348119,180327,167792,125754,59838,65916
1,ang mo kio,total,179297,87113,92184,146966,71220,75746,12873,6355,6518,14952,7532,7420,4506,2006,2500
2,NaN,cheng san,30503,14883,15620,24815,12064,12751,2437,1221,1216,2539,1291,1248,712,307,405
3,NaN,chong boon,29903,14671,15232,24613,12043,12570,2005,989,1016,2625,1346,1279,660,293,367
4,NaN,kebun bahru,25854,12513,13341,20486,9864,10622,2338,1172,1166,2331,1167,1164,699,310,389


In [24]:
# make the planning_area and subzone names lowercaps
ethnicity_2015[columns_to_lower] = ethnicity_2015[columns_to_lower].apply(lambda x: x.str.lower().str.strip(),
                                                                          axis = 0)
# remove the - total trailing string from planning_area column
ethnicity_2015['planning_area'] = ethnicity_2015['planning_area'].str.replace('- total', '').str.strip()
ethnicity_2015.head()

,planning_area,subzone,total,male_total,female_total,total_chinese,male_chinese,female_chinese,total_malays,male_malays,female_malays,total_indians,male_indians,female_indians,total_others,male_others,female_others
0,total,total,3902690,1916630,1986060,2900010,1415300,1484700,520920,259110,261820,354950,182300,172650,126810,59910,66900
1,ang mo kio,NaN,174770,84220,90550,143290,68860,74430,13060,6430,6640,14150,7050,7100,4270,1890,2390
2,NaN,ang mo kio town centre,5020,2370,2640,4260,2020,2240,210,90,120,360,170,190,190,100,90
3,NaN,cheng san,29770,14400,15370,24660,11890,12770,2140,1080,1060,2380,1210,1170,600,230,370
4,NaN,chong boon,27900,13590,14310,22910,11150,11760,1950,950,1010,2400,1230,1170,630,260,370


In [25]:
# make the planning_area and subzone names lowercaps
ethnicity_2020[columns_to_lower] = ethnicity_2020[columns_to_lower].apply(lambda x: x.str.lower().str.strip(),
                                                                          axis = 0)
# remove the - total trailing string from planning_area column
ethnicity_2020['planning_area'] = ethnicity_2020['planning_area'].str.replace('- total', '').str.strip()
ethnicity_2020.head()

,planning_area,subzone,total,male_total,female_total,total_chinese,male_chinese,female_chinese,total_malays,male_malays,female_malays,total_indians,male_indians,female_indians,total_others,male_others,female_others
0,total,total,4044210,1977560,2066650,3006770,1461340,1545430,545500,271330,274170,362270,185570,176710,129670,59320,70350
1,ang mo kio,NaN,162280,77570,84700,134350,64050,70300,11140,5480,5660,12810,6350,6470,3970,1690,2280
2,NaN,ang mo kio town centre,4810,2260,2550,4140,1950,2200,140,60,80,370,170,200,160,80,80
3,NaN,cheng san,28070,13480,14600,23240,11120,12120,1940,960,980,2280,1150,1130,610,250,370
4,NaN,chong boon,26500,12860,13640,21860,10590,11270,1830,920,900,2240,1130,1110,580,220,360


In [26]:
def compare_rows(df_one, df_one_column: str, df_two, df_two_column: str):
    """
    To compare the rows of 2 dataframes to check for missing rows or mismatch row names

    Parameters
    ------
    - df_one: dataframe
        the first dataframe you want to compare

    - df_one_column: str
        name of the columns of the first dataframe you want to compare
    
    - df_two: dataframe
        the second dataframe you want to compare

    - df_two_column: str
        name of the columns of the second dataframe you want to compare

    Returns
    -------
    - missing areas of the first subzone
    
    - missing areas of the second subzone
    """

    first_subzones = df_one[df_one_column].tolist()
    second_subzones = df_two[df_two_column].tolist()

    # removing \n
    cleaned_first_subzones = [str(s).replace('\n', '') for s in first_subzones]
    cleaned_second_subzones = [str(s).replace('\n', '') for s in second_subzones]

    missing_areas_of_first_subzone = set(cleaned_second_subzones).difference(set(cleaned_first_subzones))
    print(len(cleaned_first_subzones))
    # print(missing_areas_of_first_subzone)

    missing_areas_of_second_subzone = set(cleaned_first_subzones).difference(set(cleaned_second_subzones))
    print(len(cleaned_second_subzones))
    # print(missing_areas_of_second_subzone)

    return missing_areas_of_first_subzone, missing_areas_of_second_subzone


#### 2010 data is missing quite a lot for planning area. while 2015 and 2020 dataset have the same planning areas

In [27]:
missing_area_of_2010, missing_area_of_2015 = compare_rows(ethnicity_2010, "planning_area",
                                                          ethnicity_2015, "planning_area")
print(f"missing for 2010: {missing_area_of_2010}")
print(f"missing for 2015: {missing_area_of_2015}")

missing_area_of_2015, missing_area_of_2020 = compare_rows(ethnicity_2015, "planning_area",
                                                          ethnicity_2020, "planning_area")
print(f"missing for 2015: {missing_area_of_2015}")
print(f"missing for 2020: {missing_area_of_2020}")

228
379
missing for 2010: {'marina east', 'paya lebar', 'sungei kadut', 'simpang', 'lim chu kang', 'pioneer', 'central water catchment', 'boon lay', 'tengah', 'seletar', 'changi bay', 'north-eastern islands', 'western islands', 'marina south', 'museum', 'southern islands', 'orchard', 'tuas', 'straits view', 'western water catchment'}
missing for 2015: {'others'}
379
388
missing for 2015: set()
missing for 2020: set()


#### 2010 when compared to 2015 has quite a few subzones missing

In [28]:
missing_area_of_2010, missing_area_of_2015 = compare_rows(ethnicity_2010, "subzone",
                                                          ethnicity_2015, "subzone")
print(f"missing for 2010: {missing_area_of_2010}")
print(f"missing for 2015: {missing_area_of_2015}")

missing_area_of_2015, missing_area_of_2020 = compare_rows(ethnicity_2015, "subzone",
                                                          ethnicity_2020, "subzone")
print(f"missing for 2015: {missing_area_of_2015}")
print(f"missing for 2020: {missing_area_of_2020}")

228
379
missing for 2010: {'paya lebar north', 'waterway east', 'flora drive', 'ang mo kio town centre', 'city terminals', 'shipyard', 'pulau punggol barat', 'central subzone', 'pioneer sector', 'the wharves', 'anson', 'pasir ris central', 'east coast', "national university of s'pore", 'boon lay place', 'central water catchment', 'paya lebar west', 'kranji', 'sentosa', 'punggol town centre', 'coney island', 'tengah', 'alexandra north', 'mackenzie', 'one north', 'tukang', 'marina south', 'jelebu', 'yuhua west', 'plab', 'jurong west central', 'tampines north', 'benoi sector', 'jurong river', 'mandai east', 'defu industrial park', 'chin bee', 'compassvale', 'hougang west', 'somerset', 'reservoir view', 'marina east (mp)', 'changi airport', 'singapore general hospital', 'gul basin', 'jurong port', 'marina centre', 'toa payoh west', 'matilda', 'gali batu', 'port', 'lim chu kang', 'toh tuck', 'yuhua east', 'selegie', 'lorong 8 toa payoh', 'lorong halus north', 'changi point', 'safti', 'kalla

#### 2015 and 2020 subzone differences are similar to that in changes_in_population.ipynb

In [29]:
# subzones_to_remove_from_2020 = ["brickland", "garden", "park", "plantation",
#                       "tengah industrial estate", "bahar", "cleantech", "lakeside (business)"]

# subzone_name_mapping = {
#     'forest hill': 'tengah',
#     'murai': 'western water catchment',
#     'lakeside (leisure)': 'lakeside',
# }

In [30]:
# # Keep rows where Subzone names is NOT in the list of subzones to remove
# filtered_subzone_2020 = ethnicity_2020[
#     ~ethnicity_2020['subzone'].isin(subzones_to_remove_from_2020)
# ].copy()

# target_column = "subzone"
# # .fillna(df[target_column]) fills in NaNs (caused by unmapped values) with the original values.
# ethnicity_2020[target_column] = filtered_subzone_2020[target_column].map(subzone_name_mapping).fillna(filtered_subzone_2020[target_column])

# missing_areas_of_subzone_2015, missing_areas_of_subzone_2020 = compare_rows(ethnicity_2015, "subzone", ethnicity_2020, "subzone")

# print("subzone name missing/mismatch from 2015 population dataset: ", missing_areas_of_subzone_2015)
# print("subzone name missing/mismatch from 2020 population dataset: ", missing_areas_of_subzone_2020)

In [31]:
display(ethnicity_2015["subzone"].describe())
display(ethnicity_2020["subzone"].describe())

count       324
unique      324
top       total
freq          1
Name: subzone, dtype: object

count       333
unique      333
top       total
freq          1
Name: subzone, dtype: object

#### 2015 and 2020 had hardly any differences in planning area and subzones, except for the inclusin of nicoll in 2020

#### 2010 and 2015 had lots of differences in both planning area and subzones. 

In [32]:
def save_to_excel_with_multiple_sheets(df, filepath, sheet_name_to_update= None):
    if filepath.exists():
        excel_mode = 'a'
        writer_kwargs = {'mode': excel_mode, 'if_sheet_exists': 'replace'}
    else:
        excel_mode = 'w'
        writer_kwargs = {'mode': excel_mode}

    with pd.ExcelWriter(
        filepath, 
        engine='openpyxl',
        **writer_kwargs # Unpack the dictionary of arguments
    ) as writer:
        # Write the DataFrame to the specified sheet
        df.to_excel(
            writer, 
            sheet_name=sheet_name_to_update, 
            index=False # Set to True if you want to save the DataFrame index
        )
    print(f"Save to filepath: {filepath}" )

#### Some data cleaning up

In [33]:
# fill the empty cells in subzone with "total" string
ethnicity_2010['subzone'] = ethnicity_2010['subzone'].fillna('total')
ethnicity_2015['subzone'] = ethnicity_2015['subzone'].fillna('total')
ethnicity_2020['subzone'] = ethnicity_2020['subzone'].fillna('total')

# replace string instances of na and - into np.nan
ethnicity_2010 = (
    ethnicity_2010.replace(MISSING_VALUES, np.nan, regex=False))
ethnicity_2015 = (
    ethnicity_2015.replace(MISSING_VALUES, np.nan, regex=False))
ethnicity_2020 = (
    ethnicity_2020.replace(MISSING_VALUES, np.nan, regex=False))

## fill the empty cells with the planning area names so that it is easier to work with
ethnicity_2010['planning_area'] = ethnicity_2010['planning_area'].replace('', pd.NA)
# propagate the last valid 'planning_area' value downward.
ethnicity_2010['planning_area'] = ethnicity_2010['planning_area'].ffill()

## fill the empty cells with the planning area names so that it is easier to work with
ethnicity_2015['planning_area'] = ethnicity_2015['planning_area'].replace('', pd.NA)
# propagate the last valid 'planning_area' value downward.
ethnicity_2015['planning_area'] = ethnicity_2015['planning_area'].ffill()

## fill the empty cells with the planning area names so that it is easier to work with
ethnicity_2020['planning_area'] = ethnicity_2020['planning_area'].replace('', pd.NA)
# propagate the last valid 'planning_area' value downward.
ethnicity_2020['planning_area'] = ethnicity_2020['planning_area'].ffill()

# lastly convert the columns with numbers into int64
convert_cols_to_int(ethnicity_2010)
convert_cols_to_int(ethnicity_2015)
convert_cols_to_int(ethnicity_2020)

/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_5505/3675869346.py:10: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ethnicity_2015.replace(MISSING_VALUES, np.nan, regex=False))
/var/folders/b8/1t_kfq0s27zbtp79vq8klgfm0000gn/T/ipykernel_5505/3675869346.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  ethnicity_2020.replace(MISSING_VALUES, np.nan, regex=False))


In [34]:
# saving the cleaned ethnicity dataframes
OUTPUT_FILEPATH = Path(BASE_DATASET_PATH / "singapore_data/cleaned_data/ethnicity_combined.xlsx")

save_to_excel_with_multiple_sheets(ethnicity_2010, OUTPUT_FILEPATH, sheet_name_to_update = "2010")
save_to_excel_with_multiple_sheets(ethnicity_2015, OUTPUT_FILEPATH, sheet_name_to_update = "2015")
save_to_excel_with_multiple_sheets(ethnicity_2020, OUTPUT_FILEPATH, sheet_name_to_update = "2020")

Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/ethnicity_combined.xlsx
Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/ethnicity_combined.xlsx
Save to filepath: /Users/yitong/Desktop/Uni_Skool/Y3 Internship SingHealth/code/Mobile-AED-Study/datasets/singapore_data/cleaned_data/ethnicity_combined.xlsx
